# Equation 1 — Matched Pegged-Base Countries Comparison

Replication code for the baseline difference-in-differences regression from:

> Albertazzi, U., 't Hooft, J., & ter Steege, L. (2025).  
> *The causal effect of inflation on financial stability: evidence from history.*  
> ECB Working Paper Series No. 3108.

---

## Methodology

The key identification challenge is that monetary policy — itself a well-documented driver of financial crises — reacts endogenously to inflation. To isolate inflation's **direct** effect on financial stability we exploit the **monetary trilemma**: a country with a pegged exchange rate and open capital markets is forced to import the monetary policy stance of its base country, irrespective of domestic inflation conditions.

This lets us compare two types of country that share the same monetary policy shock:

| | BASE country | PEG country |
|---|---|---|
| Monetary policy | Set in response to **domestic** inflation | Imported from BASE |
| Inflation shock | Domestic | May differ from BASE |

The key variable is the interaction $\Delta_4\pi^{\text{trilemma}}_{i,t-1} \times \text{BASE}_{it}$, which captures the extra crisis risk that accrues to BASE countries, over and above what the matched PEG country experiences, following a domestic inflationary shock. Because both countries face the same interest rate path, any remaining difference in crisis probability must reflect the **direct** effect of inflation.

The regression estimated is (Equation 1 in the paper):

$$
\text{crisis}_{i,(t,t+2)} = \alpha_i + \beta_0
  + \beta_1 \, \Delta_4\pi^{\text{trilemma}}_{i,t-1} \times \text{BASE}_{it}
  + \beta_2 \, \text{BASE}_{it}
  + \beta_3 \, \Delta_4\pi^{\text{trilemma}}_{i,t-1}
  + \sum_{j=0}^{4} \beta_{4,j} \Delta y_{i,t-j}
  + \sum_{j=0}^{4} \beta_{5,j} \Delta\pi_{i,t-j}
  + \epsilon_{it}
$$

where $\text{crisis}_{i,(t,t+2)} = 1$ if a systemic banking crisis begins in country $i$ in year $t$, $t+1$, or $t+2$.  Standard errors are clustered at the country level.

## 1. Imports

In [21]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
from utils import melt, add_const, winsor, display_w_std_errors
from data import load_jst, load_kaopen

## 2. Data

### 2.1 Jordà-Schularick-Taylor Macrohistory Database

Annual macro-financial data for 18 advanced economies, 1870–2020.  
Source: Jordà, Schularick & Taylor (2017); dataset R6 available at [www.macrohistory.net/database](https://www.macrohistory.net/database/).

In [22]:
raw_df = load_jst()
def pivot(v): return raw_df.pivot(index='year', columns='country', values=v)

### 2.2 Chinn-Ito Capital Openness Index (KAOPEN)

Standardised measure of capital account openness (0 = closed, 1 = fully open).  
Source: Chinn & Ito (2006); data available at [web.pdx.edu/~ito/Chinn-Ito_website.htm](http://web.pdx.edu/~ito/Chinn-Ito_website.htm).

In [23]:
kaopen_df = load_kaopen()

## 3. Variable Construction

Pivot all series needed for the regression into wide (year × country) format.

In [24]:
# ── Core macro series ──────────────────────────────────────────────────────────
cpi_df    = pivot('cpi')        # consumer price index (level)
gdp_df    = pivot('rgdpbarro')  # real GDP per capita (Barro-Ursúa)
crisis_df = pivot('crisisJST')  # systemic banking crisis dummy (JST definition)

# ── Currency peg data ──────────────────────────────────────────────────────────
peg_df  = pivot('peg')      # =1 if country has a pegged exchange rate
base_df = pivot('peg_base') # ISO code of the base currency country

## 4. Identifying BASE and PEG Countries

A country is classified as a **BASE** country in year $t$ if another country's currency is pegged to it.  The main base currencies in the sample are GBR (gold standard era), USA (post-Bretton Woods), DEU/FRA (European exchange rate arrangements), and a FRA-USA hybrid used for some country-year pairs.

In [25]:
# Initialise BASE indicator to zero for all country-years
is_base_df = cpi_df.copy()
is_base_df[:] = 0

# Mark each country as BASE in years when other countries peg to it
for year in range(1870, 2021):
    for base_country in base_df.loc[year].unique():
        if base_country in ['GBR', 'FRA', 'USA', 'HYBRID', 'DEU']:
            is_base_df.loc[year, base_country] = 1

is_base_df = is_base_df.drop('HYBRID', axis=1)

# Combined weight: a PEG country's effective transmission of the BASE country's
# monetary policy is proportional to its capital account openness (KAOPEN).
# BASE countries always receive weight 1.
kaopen_or_base_df = (is_base_df + kaopen_df).clip(upper=1)

## 5. Constructing the Trilemma Inflation Variable

For each country $i$ and year $t$ we define:

$$
\Delta\pi^{\text{trilemma}}_{i,t} =
\begin{cases}
\Delta\pi_{i,t} & \text{if } i \text{ is a BASE country} \\
\Delta\pi_{\text{base},t} \times \text{KAOPEN}_{i,t} & \text{if } i \text{ is a PEG country}
\end{cases}
$$

This variable captures the monetary policy reaction to inflation in the BASE country that is transmitted to the PEG country in proportion to its degree of capital mobility.

In [26]:
def base_country_data(target_df_input):
    """
    For each country-year, return the inflation of the relevant base currency country.
    
    BASE countries: return their own inflation (used directly in the interaction term).
    PEG countries:  return the base country's inflation (to be scaled by KAOPEN later).
    
    A 'HYBRID' base (equal weight FRA + USA) is handled explicitly for the small
    number of country-years where no single base country applies.
    
    Parameters
    ----------
    target_df_input : pd.DataFrame
        Wide DataFrame (year × country) of any variable — typically inflation changes.
    
    Returns
    -------
    pd.DataFrame
        Wide DataFrame with the same shape, filled with the base country's value.
    """
    target_df = target_df_input.copy()
    target_df['HYBRID'] = (target_df['FRA'] + target_df['USA']) / 2
    base_target_df = target_df.copy()
    base_target_df[:] = np.nan
    for country in base_df.columns:
        for year in range(1870, 2021):
            if is_base_df.loc[year, country] == 1:
                base_country = country
            else:
                base_country = base_df.loc[year, country]
            if base_country in ['GBR', 'FRA', 'USA', 'HYBRID', 'DEU']:
                base_target_df.loc[year, country] = target_df.loc[year, base_country]
    return base_target_df


# 4-year change in CPI inflation (winsorised), computed for both own and base countries
inflation_delta      = winsor((cpi_df.pct_change() * 100).diff(4))
base_inflation_delta = base_country_data(inflation_delta) * kaopen_or_base_df

## 6. Building the Regression Panel

We construct a (country, year) panel with:
- **Dependent variable**: `crisis` — equals 1 if a crisis begins in year $t$, $t+1$, or $t+2$
- **Key regressor**: the trilemma inflation interaction ($\Delta_4\pi^{\text{trilemma}} \times \text{BASE}$)
- **Controls**: five lags each of annual inflation changes and real GDP per capita growth

The sample excludes WWI (1914–1918) and WWII (1939–1945), and retains only country-years where the country was a BASE or PEG throughout the lag window (ensuring the trilemma identification assumption holds continuously).

In [27]:
panel_df = raw_df.set_index(['country', 'year'])
panel_df['year'] = panel_df.reset_index()['year'].values

# ── Dependent variable: crisis within a 3-year forward window ──────────────────
crises_across_horizon_df = (
    (crisis_df == 1) | (crisis_df.shift(-1) == 1) | (crisis_df.shift(-2) == 1)
)
panel_df['crisis'] = melt(crises_across_horizon_df)

# ── Controls: lags 0–4 of annual inflation change and GDP growth ───────────────
infl_lag   = 1   # lag at which the trilemma interaction enters the regression
n_lags     = 4   # number of control lags
covariates = []

for lag in range(0, n_lags + 1):
    panel_df[f'infl_d({lag})'] = melt(
        winsor((cpi_df.pct_change() * 100).diff()).shift(lag))
    covariates.append(f'infl_d({lag})')
    panel_df[f'gdp({lag})'] = melt(
        winsor(gdp_df.pct_change(fill_method=None) * 100).shift(lag))
    covariates.append(f'gdp({lag})')

# ── Key regressors ─────────────────────────────────────────────────────────────
# β₁: Δ₄π^trilemma × BASE  — the causal effect of inflation net of monetary policy
panel_df[f'delta_lag_{infl_lag}_interaction'] = melt(
    (is_base_df * base_inflation_delta).shift(infl_lag))
# β₃: Δ₄π^trilemma (uninteracted) — effect common to both BASE and PEG countries
panel_df[f'infl_delta_lag_{infl_lag}'] = melt(base_inflation_delta.shift(infl_lag))
# β₂: BASE dummy — absorbs any level difference between BASE and PEG countries
panel_df[f'base_lag_{infl_lag}'] = melt(is_base_df.shift(infl_lag))

x_variables = [
    f'delta_lag_{infl_lag}_interaction',
    f'infl_delta_lag_{infl_lag}',
    f'base_lag_{infl_lag}',
] + covariates

# ── Sample restrictions ────────────────────────────────────────────────────────
is_and_was_base_or_peg_df = (
    (peg_df.rolling(infl_lag + 1).min().fillna(0) == 1) |
    (is_base_df.rolling(infl_lag + 1).min().fillna(0) == 1)
)
panel_df['is_and_was_base_or_peg'] = melt(is_and_was_base_or_peg_df)

sub_sample = panel_df[(panel_df.year >= 1870) & (panel_df.year < 2020)]
war_years  = list(range(1914, 1919)) + list(range(1939, 1946))
sub_sample = sub_sample[~sub_sample.year.isin(war_years)]
sub_sample = sub_sample[sub_sample.is_and_was_base_or_peg]

## 7. Estimation

Linear probability model with country fixed effects and country-clustered standard errors, as in Equation 1 of the paper.  The coefficient of interest is `delta_lag_1_interaction` ($\hat{\beta}_1$).

In [28]:
regression_df = sub_sample[['crisis'] + x_variables].dropna()

fit = PanelOLS(
    dependent      = regression_df['crisis'],
    exog           = add_const(regression_df[x_variables]),
    entity_effects = True,
    time_effects   = False,
).fit(cov_type='clustered', cluster_entity=True)

# Display the three key coefficients (interaction, base inflation, BASE dummy)
display_w_std_errors(fit).iloc[1:4]

,results_w_errors
delta_lag_1_interaction,0.012*** (0.003)
infl_delta_lag_1,-0.009** (0.004)
base_lag_1,-0.084*** (0.011)
